# Parameter Grid Search

This notebook demonstrates `lsp.grid_search()` to systematically test Suite2p/Cellpose parameter combinations.

**Key Features:**
- Automatically detects registration vs detection parameters
- When searching only detection parameters, binary is written and registered once, then reused
- Supports raw data with frame limiting via `reader_kwargs`
- Supports plane selection for volumetric data

## Workflow

1. Specify parameters to search with `grid_params`
2. Optionally provide base `ops` dictionary with fixed parameters
3. `grid_search` tests all combinations and saves results to separate folders

In [1]:
import numpy as np
from pathlib import Path
import lbm_suite2p_python as lsp

data_dir = Path(r"D:/demo/raw")           # directory containing input files
save_path = Path(r"D:/demo/grid_search")  # output directory

# for large raw data, you can limit frames:
# reader_kwargs = {"frames": slice(0, 1500)}

## Define Grid Search Parameters

Choose which parameters to search. 

**Detection parameters** (fast - registration runs once):

| Parameter | Description | Typical Range |
|-----------|-------------|---------------|
| `threshold_scaling` | Detection threshold multiplier | 0.5 - 2.0 |
| `diameter` | Expected cell diameter in pixels | 4 - 15 px |
| `spatial_hp_cp` | High-pass for Cellpose (× diameter) | 0 - 3 |
| `anatomical_only` | Cellpose mode (2=mean, 3=enhanced) | 2, 3, 4 |
| `sparse_mode` | Suite2p sparse detection | True, False |

**Registration parameters** (slow - each combination re-registers):

| Parameter | Description |
|-----------|-------------|
| `nonrigid` | Enable non-rigid registration |
| `block_size` | Non-rigid block size |
| `maxregshift` | Max registration shift (fraction) |
| `smooth_sigma` | Spatial smoothing for registration |

In [2]:
# Grid search high-pass filter values
grid_params = {
    "spatial_hp_cp": [0, 0.5, 3, 10],
}

# Show combinations
n_combos = np.prod([len(v) for v in grid_params.values()])
print(f"Parameters: {list(grid_params.keys())}")
print(f"Total combinations: {n_combos}")

Parameters: ['spatial_hp_cp']
Total combinations: 4


## Run Grid Search

Each parameter combination is saved to a separate subdirectory.

When searching **only detection parameters**, the binary is written once to `_base/` and registration runs once. Then detection runs for each combination using the same registered binary - much faster!

When searching **registration parameters**, each combination requires full re-registration.

In [3]:
import mbo_utilities as mbo
# base ops (fixed parameters for all combinations)
ops = mbo.metadata.default_ops()
ops["anatomical_only"] = 4
ops["diameter"] = 4
ops["use_builtin_classifier"] = 4

# run grid search
# since spatial_hp_cp is a detection param, registration only runs once
lsp.grid_search(
    input_data=data_dir,
    save_path=save_path,
    grid_params=grid_params,
    ops=ops,
    planes=7,  # optional: only process plane 7 from volumetric data
    # reader_kwargs={"frames": slice(0, 1500)},  # optional: limit frames for large raw data
    force_reg=False,    # skip registration if already done
    force_detect=True,  # re-run detection for each combination
)

Importing suite2p packages...
Only detection parameters in grid - will reuse registered binary.
Grid search: D:\demo\grid_search
Parameters: ['spatial_hp_cp']

Loading input data...
  Input: D:\demo\raw


Counting frames:   0%|          | 0/2 [00:00<?, ?it/s]

  Loaded as: MboRawArray

Dataset info:
  Shape: (1574, 14, 550, 448)
  Frames: 1574
  Planes: 14
  Dimensions: 550 x 448
  Processing planes: [7]

Total combinations: 4

Processing plane 7
Importing suite2p packages...

Using existing base binary: D:\demo\grid_search\_base\data_raw.bin
Using existing registered binary: D:\demo\grid_search\_base\data.bin

[1/4] spa0
Importing suite2p packages...
  Running detection...
Registration skipped - copying data_raw.bin to data.bin...
  data.bin already exists, skipping copy
  Using full image dimensions for anatomical detection (avoids cropping issues)
NOTE: applying default C:\Users\RBO\.suite2p\classifiers\classifier_user.npy
----------- ROI DETECTION
Binning movie in chunks of length 18
Binned movie of size [87,550,448] created in 1.01 sec.


model_type argument is not used in v4.0.1+. Ignoring this argument...


>>>> CELLPOSE finding masks in max_proj
!NOTE! diameter set to 4.00 for cell detection with cellpose


...

>>>> 0 masks detected, median diameter = nan 
Detected 0 ROIs, 14.22 sec
----------- Total 15.13 sec.
no ROIs found, only ops.npy file saved
  Failed to generate quality diagnostics: Missing required files in D:\demo\grid_search\spa3: F.npy, Fneu.npy, spks.npy, stat.npy, iscell.npy
  Failed to generate regional zoom: Missing required files in D:\demo\grid_search\spa3: F.npy, Fneu.npy, spks.npy, stat.npy, iscell.npy

[4/4] spa10
Importing suite2p packages...
  Running detection...
Registration skipped - copying data_raw.bin to data.bin...
  data.bin already exists, skipping copy
  Using full image dimensions for anatomical detection (avoids cropping issues)
NOTE: applying default C:\Users\RBO\.suite2p\classifiers\classifier_user.npy
----------- ROI DETECTION
Binning movie in chunks of length 18


model_type argument is not used in v4.0.1+. Ignoring this argument...


Binned movie of size [87,550,448] created in 0.92 sec.
>>>> CELLPOSE finding masks in max_proj
!NOTE! diameter set to 4.00 for cell detection with cellpose


channels deprecated in v4.0.1+. If data contain more than 3 channels, only the first 3 channels will be used
no seeds found in get_masks_torch - no masks found.


>>>> 0 masks detected, median diameter = nan 
Detected 0 ROIs, 14.53 sec
----------- Total 15.51 sec.
no ROIs found, only ops.npy file saved
  Failed to generate quality diagnostics: Missing required files in D:\demo\grid_search\spa10: F.npy, Fneu.npy, spks.npy, stat.npy, iscell.npy
  Failed to generate regional zoom: Missing required files in D:\demo\grid_search\spa10: F.npy, Fneu.npy, spks.npy, stat.npy, iscell.npy

Grid search complete: 4 combinations
Results in: D:\demo\grid_search


## Analyze Results

Collect quality metrics from all runs: SNR, skewness, and shot noise distributions.

In [4]:
import pandas as pd
from scipy.stats import skew

def compute_quality_metrics(F, Fneu, stat, iscell, fs=30.0):
    """Compute SNR, skewness, and shot noise for accepted cells."""
    mask = iscell[:, 0].astype(bool)
    n_accepted = mask.sum()

    if n_accepted == 0:
        return {
            "n_accepted": 0, "n_rejected": len(mask),
            "snr_median": np.nan, "snr_iqr": np.nan,
            "skew_median": np.nan, "skew_iqr": np.nan,
            "noise_median": np.nan, "noise_iqr": np.nan,
        }

    F_acc = F[mask]
    Fneu_acc = Fneu[mask]
    stat_acc = [s for s, m in zip(stat, mask) if m]

    # neuropil-corrected fluorescence
    F_corr = F_acc - 0.7 * Fneu_acc

    # baseline (20th percentile)
    baseline = np.percentile(F_corr, 20, axis=1, keepdims=True)
    baseline = np.maximum(baseline, 1e-6)

    # ΔF/F
    dff = (F_corr - baseline) / baseline

    # SNR: signal (std of dff) / noise (MAD estimator)
    signal = np.std(dff, axis=1)
    noise = np.median(np.abs(np.diff(dff, axis=1)), axis=1) / 0.6745
    snr = signal / (noise + 1e-6)

    # shot noise: MAD of frame differences, normalized by sqrt(framerate)
    shot_noise = np.median(np.abs(np.diff(dff, axis=1)), axis=1) / np.sqrt(fs)

    # skewness: from stat if available, else compute
    skewness = []
    for i, s in enumerate(stat_acc):
        if "skew" in s:
            skewness.append(s["skew"])
        else:
            skewness.append(skew(dff[i]))
    skewness = np.array(skewness)

    return {
        "n_accepted": n_accepted,
        "n_rejected": len(mask) - n_accepted,
        "snr_median": np.median(snr),
        "snr_iqr": np.percentile(snr, 75) - np.percentile(snr, 25),
        "skew_median": np.median(skewness),
        "skew_iqr": np.percentile(skewness, 75) - np.percentile(skewness, 25),
        "noise_median": np.median(shot_noise),
        "noise_iqr": np.percentile(shot_noise, 75) - np.percentile(shot_noise, 25),
    }

def find_ops_file(combo_dir):
    """Find ops.npy in combo directory, checking common locations."""
    # direct location (new grid_search output)
    ops_file = combo_dir / "ops.npy"
    if ops_file.exists():
        return ops_file

    # check for subdirectories (legacy output or run_plane subdir)
    for subdir in combo_dir.iterdir():
        if subdir.is_dir():
            ops_file = subdir / "ops.npy"
            if ops_file.exists():
                return ops_file
    return None

# collect results
results = []

for combo_dir in save_path.iterdir():
    if not combo_dir.is_dir():
        continue

    # skip non-parameter directories
    if combo_dir.name in ("grid_search_results.csv", "_base"):
        continue

    ops_file = find_ops_file(combo_dir)
    if ops_file is None:
        continue

    try:
        res = lsp.load_planar_results(ops_file)
        loaded_ops = lsp.load_ops(ops_file)
        fs = loaded_ops.get("fs", 30.0)

        # compute quality metrics
        metrics = compute_quality_metrics(
            res["F"], res["Fneu"], res["stat"], res["iscell"], fs
        )

        # use outer directory name (the grid search tag)
        result = {"combo": combo_dir.name, "ops_file": str(ops_file), **metrics}

        # add grid search parameters
        for param in grid_params.keys():
            result[param] = loaded_ops.get(param)

        results.append(result)

    except Exception as e:
        print(f"Skipping {combo_dir.name}: {e}")

# create DataFrame
if results:
    df = pd.DataFrame(results)

    # sort by SNR (higher is better)
    df = df.sort_values("snr_median", ascending=False)

    print("Quality Metrics (sorted by median SNR):")
    print("=" * 100)
    display_cols = ["combo", "n_accepted", "snr_median", "skew_median", "noise_median"] + list(grid_params.keys())
    print(df[display_cols].to_string(index=False))

    # save
    csv_path = save_path / "grid_search_results.csv"
    df.to_csv(csv_path, index=False)
    print(f"\nSaved to: {csv_path}")
else:
    print("No results found!")
    df = pd.DataFrame()

Skipping spa0: Missing required files in D:\demo\grid_search\spa0: F.npy, Fneu.npy, spks.npy, stat.npy, iscell.npy
Skipping spa0.50: Missing required files in D:\demo\grid_search\spa0.50: F.npy, Fneu.npy, spks.npy, stat.npy, iscell.npy
Skipping spa10: Missing required files in D:\demo\grid_search\spa10: F.npy, Fneu.npy, spks.npy, stat.npy, iscell.npy
Skipping spa3: Missing required files in D:\demo\grid_search\spa3: F.npy, Fneu.npy, spks.npy, stat.npy, iscell.npy
No results found!


## Visualize Quality Metrics

Compare SNR, skewness, and shot noise distributions across parameter combinations.

In [5]:
import matplotlib.pyplot as plt

if len(df) > 0:
    plt.style.use('dark_background')

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))

    metrics = [
        ("snr_median", "SNR (median)", "higher is better", "#2ecc71"),
        ("skew_median", "Skewness (median)", "higher = more events", "#9b59b6"),
        ("noise_median", "Shot Noise (median)", "lower is better", "#e74c3c"),
    ]

    # Row 1: Metrics by parameter
    params = list(grid_params.keys())

    for col, (metric, label, note, color) in enumerate(metrics):
        ax = axes[0, col]

        # Bar chart of metric values
        x = range(len(df))
        ax.bar(x, df[metric], color=color, alpha=0.8, edgecolor="white")
        ax.set_xticks(x)
        ax.set_xticklabels(df["combo"], rotation=45, ha="right", fontsize=8)
        ax.set_ylabel(label)
        ax.set_title(f"{label}\n({note})")
        ax.axhline(df[metric].median(), color="white", linestyle="--", alpha=0.5, label="median")

    # Row 2: Parameter effects on SNR
    for col, param in enumerate(params[:3]):  # Show up to 3 params
        ax = axes[1, col]

        grouped = df.groupby(param)["snr_median"].agg(["mean", "std"]).reset_index()
        x = range(len(grouped))
        ax.bar(x, grouped["mean"], yerr=grouped["std"],
               color="#3498db", alpha=0.8, edgecolor="white", capsize=5)
        ax.set_xticks(x)
        ax.set_xticklabels([str(v) for v in grouped[param]])
        ax.set_xlabel(param, fontweight="bold")
        ax.set_ylabel("SNR (mean ± std)")
        ax.set_title(f"Effect of {param} on SNR")

    # Hide unused subplots
    for col in range(len(params), 3):
        axes[1, col].axis("off")

    plt.suptitle("Grid Search Quality Metrics", fontsize=14, fontweight="bold", y=1.02)
    plt.tight_layout()

    fig_path = save_path / "quality_metrics.png"
    plt.savefig(fig_path, dpi=150, bbox_inches="tight", facecolor="black")
    print(f"Saved: {fig_path}")
    plt.show()

## Distribution Comparison

Compare the full distributions of SNR, skewness, and shot noise for top combinations.

In [6]:
if len(df) > 0:
    # get top 4 by SNR
    top_combos = df.head(4)["combo"].tolist()

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    colors = ["#2ecc71", "#3498db", "#e74c3c", "#f39c12"]

    for combo, color in zip(top_combos, colors):
        combo_dir = save_path / combo
        if combo == "_base":  # skip base directory
            continue
        ops_file = find_ops_file(combo_dir)
        if ops_file is None:
            continue

        try:
            res = lsp.load_planar_results(ops_file)
            loaded_ops = lsp.load_ops(ops_file)
            fs = loaded_ops.get("fs", 30.0)

            mask = res["iscell"][:, 0].astype(bool)
            if mask.sum() == 0:
                continue

            F_acc = res["F"][mask]
            Fneu_acc = res["Fneu"][mask]
            stat_acc = [s for s, m in zip(res["stat"], mask) if m]

            F_corr = F_acc - 0.7 * Fneu_acc
            baseline = np.percentile(F_corr, 20, axis=1, keepdims=True)
            baseline = np.maximum(baseline, 1e-6)
            dff = (F_corr - baseline) / baseline

            # SNR
            signal = np.std(dff, axis=1)
            noise = np.median(np.abs(np.diff(dff, axis=1)), axis=1) / 0.6745
            snr = signal / (noise + 1e-6)

            # shot noise
            shot_noise = np.median(np.abs(np.diff(dff, axis=1)), axis=1) / np.sqrt(fs)

            # skewness
            skewness = np.array([s.get("skew", skew(dff[i])) for i, s in enumerate(stat_acc)])

            # plot distributions
            axes[0].hist(snr, bins=30, alpha=0.5, color=color, label=combo, density=True)
            axes[1].hist(skewness, bins=30, alpha=0.5, color=color, label=combo, density=True)
            axes[2].hist(shot_noise, bins=30, alpha=0.5, color=color, label=combo, density=True)

        except Exception as e:
            print(f"Error loading {combo}: {e}")

    axes[0].set_xlabel("SNR")
    axes[0].set_ylabel("Density")
    axes[0].set_title("SNR Distribution")
    axes[0].legend(fontsize=8)

    axes[1].set_xlabel("Skewness")
    axes[1].set_ylabel("Density")
    axes[1].set_title("Skewness Distribution")

    axes[2].set_xlabel("Shot Noise")
    axes[2].set_ylabel("Density")
    axes[2].set_title("Shot Noise Distribution")

    plt.suptitle("Quality Metric Distributions (Top 4 by SNR)", fontweight="bold", y=1.02)
    plt.tight_layout()

    fig_path = save_path / "metric_distributions.png"
    plt.savefig(fig_path, dpi=150, bbox_inches="tight", facecolor="black")
    print(f"Saved: {fig_path}")
    plt.show()

## Compare Detection Masks

In [7]:
if len(df) > 0:
    # Top 4 by SNR
    top_combos = df.head(4)

    fig, axes = plt.subplots(2, 2, figsize=(12, 12))
    axes = axes.flatten()

    for ax, (_, row) in zip(axes, top_combos.iterrows()):
        combo_dir = save_path / row["combo"]
        ops_file = find_ops_file(combo_dir)

        if ops_file is not None:
            loaded_ops = lsp.load_ops(ops_file)
            mean_img = loaded_ops.get("meanImg", loaded_ops.get("refImg", np.zeros((512, 512))))

            ax.imshow(mean_img, cmap="gray",
                      vmin=np.percentile(mean_img, 1),
                      vmax=np.percentile(mean_img, 99))

            # Draw ROIs
            stat_file = ops_file.parent / "stat.npy"
            iscell_file = ops_file.parent / "iscell.npy"

            if stat_file.exists() and iscell_file.exists():
                stat = np.load(stat_file, allow_pickle=True)
                iscell = np.load(iscell_file)[:, 0].astype(bool)

                for i, s in enumerate(stat):
                    if iscell[i]:
                        ax.scatter(s["xpix"], s["ypix"], s=0.1, c="lime", alpha=0.3)

            title = f"{row['combo']}\n{row['n_accepted']} cells, SNR={row['snr_median']:.2f}"
            ax.set_title(title, fontsize=10)

        ax.axis("off")

    plt.suptitle("Top 4 by SNR", fontsize=14, fontweight="bold")
    plt.tight_layout()

    fig_path = save_path / "detection_comparison.png"
    plt.savefig(fig_path, dpi=150, bbox_inches="tight", facecolor="black")
    print(f"Saved: {fig_path}")
    plt.show()

## Best Parameters Summary

In [8]:
if len(df) > 0:
    print("Best Parameters by Different Criteria:")
    print("=" * 60)

    # Best by SNR (highest)
    best_snr = df.loc[df["snr_median"].idxmax()]
    print(f"\nHighest SNR: {best_snr['combo']}")
    print(f"  SNR: {best_snr['snr_median']:.3f}")
    print(f"  Cells: {best_snr['n_accepted']}")
    for p in grid_params:
        print(f"  {p}: {best_snr[p]}")

    # Best by skewness (highest = more events)
    best_skew = df.loc[df["skew_median"].idxmax()]
    print(f"\nHighest Skewness: {best_skew['combo']}")
    print(f"  Skewness: {best_skew['skew_median']:.3f}")
    print(f"  Cells: {best_skew['n_accepted']}")

    # Best by noise (lowest)
    best_noise = df.loc[df["noise_median"].idxmin()]
    print(f"\nLowest Shot Noise: {best_noise['combo']}")
    print(f"  Noise: {best_noise['noise_median']:.4f}")
    print(f"  Cells: {best_noise['n_accepted']}")

## Output Structure

When searching **detection parameters only** (registration reused):

```
grid_search/
├── _base/                       # shared binary and registration
│   ├── data_raw.bin             # raw binary (written once)
│   ├── data.bin                 # registered binary (registered once)
│   └── ops.npy                  # base ops with registration results
├── grid_search_results.csv      # all metrics
├── quality_metrics.png          # metric comparison
├── metric_distributions.png     # distribution histograms
├── detection_comparison.png     # mask visualization
├── spa0.00/                     # detection-only results
│   ├── data.bin                 # copied from _base
│   ├── ops.npy
│   ├── stat.npy
│   └── F.npy
├── spa0.50/
├── spa3.00/
└── spa10.00/
```

When searching **registration parameters**:

```
grid_search/
├── non1_blo128/                 # each combo has full registration
│   ├── data_raw.bin
│   ├── data.bin
│   ├── ops.npy
│   └── stat.npy
├── non1_blo256/
└── non0_blo128/
```

## Next Steps

1. Review `grid_search_results.csv` - sort by desired metric
2. Use best parameters in `lsp.pipeline()` for full processing
3. Consider trade-offs: higher SNR vs more cells vs lower noise

## Example: Using Best Parameters

```python
# after finding best params from grid search
best_params = {
    "spatial_hp_cp": 0.5,
    "diameter": 6,
    "anatomical_only": 3,
}

# run full volume with best parameters
lsp.pipeline(
    input_data="D:/data/raw_250k_frames",
    save_path="D:/results/full_run",
    ops=best_params,
)
```